# 84 Answer time, question by question (A100)

Times the whole answer path the way a user would see it, for 100 random dev questions asked one at a time:

1. **generate**: the generator's own answer plus 16 samples, in one batched call;
2. **run**: every distinct candidate compiled and run on the database in parallel, with the 1 s cap (starts right after generating);
3. **judge**: the judge rates every candidate (one token each, sharing the schema);
4. **choose**: every method of notebook 83.

One vLLM engine holds the base model **once** with both adapters (generator and judge), so this is the single-model system as it would be deployed. The first 3 questions are left out of the summary (warm-up).

About **15 minutes, about 3 units**. Resumable. Result: `latency_v1/latency_dev.txt`.

**After every code update, reopen this notebook from the GitHub link.**

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = True  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
import json, os
R = f"{DRIVE_ROOT}/effsql/runs"
JUDGE = "judge_v1"
assert json.load(open(f"{R}/{JUDGE}/judge_meta.json"))["complete"]
!nvidia-smi --query-gpu=name,memory.total --format=csv
print("CPU cores:", os.cpu_count())

### 1. Stage the dev databases

In [ ]:
!effsql stage --split dev

### 2. Answer 100 dev questions one at a time

In [ ]:
!python -m effsql.judge latency --judge {JUDGE} --out latency_v1 --split dev

### Done: disconnect now

*Runtime > Disconnect and delete runtime.* An idle GPU keeps using compute units.